In [ ]:
# Cellule 1: Importations
import sys
from pathlib import Path

# Ajout du chemin src
sys.path.append(str(Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.data.load_data import CICDDataLoader
from src.data.preprocess import CICDPreprocessor
from src.models.detection_models import AnomalyDetector

%matplotlib inline

# Cellule 2: Chargement
loader = CICDDataLoader(Path("../data"))
# MODIFIER AVEC VOTRE NOM DE FICHIER
df = loader.load_metrics("compute_dataset.csv")
print(f"Données chargées: {df.shape}")

# Cellule 3: Prétraitement
preprocessor = CICDPreprocessor(missing_strategy='knn')
X_processed = preprocessor.fit_transform(df)
print(f"Après prétraitement: {X_processed.shape}")

# Cellule 4: Split train/test
X_train, X_test = train_test_split(X_processed, test_size=0.3, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Cellule 5: Entraînement Isolation Forest
detector = AnomalyDetector(model_type='isolation_forest', contamination=0.1)
detector.build_model()
detector.train(X_train.values)

# Cellule 6: Prédiction
y_pred = detector.predict(X_test.values)
anomalies = np.sum(y_pred == -1)
print(f"Anomalies détectées: {anomalies} ({100*anomalies/len(y_pred):.1f}%)")

# Cellule 7: Visualisation (si 2D ou avec PCA)
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_test.values)

plt.figure(figsize=(10, 6))
plt.scatter(X_pca[y_pred == 1, 0], X_pca[y_pred == 1, 1], 
            c='green', label='Normal', alpha=0.6)
plt.scatter(X_pca[y_pred == -1, 0], X_pca[y_pred == -1, 1], 
            c='red', label='Anomalie', alpha=0.8)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Détection d\'anomalies - Isolation Forest (visualisation PCA)')
plt.legend()
plt.show()

# Cellule 8: Top caractéristiques des anomalies (optionnel)
# Analyse des valeurs extrêmes
anomaly_indices = np.where(y_pred == -1)[0]
anomalies_data = X_test.iloc[anomaly_indices]
normal_data = X_test.iloc[np.where(y_pred == 1)[0]]

print("Comparaison des moyennes (anomalies vs normal):")
comparison = pd.DataFrame({
    'normale': normal_data.mean(),
    'anomalie': anomalies_data.mean(),
    'différence': anomalies_data.mean() - normal_data.mean()
})
print(comparison.sort_values('différence', key=abs, ascending=False).head(10))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("✅ Tous les paquets sont prêts!")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")